In [1]:
import pandas as pd
import requests
import datetime
import sys
# path为tools的上层文件夹
sys.path.append('D:\milimili')
from tools.数数api import liucun_download
from works.adjust_系列支出 import  get_cost
from BI_robot.feishu import FeiShu
import numpy as np
feishu=FeiShu()
import json

In [2]:
# 获取数据 两个月内的em用户新增数据、内购数据
url = 'https://dash.engageminds.ai/api/oka/analysis/v1/event/report'
headers = {
        'Content-Type': 'application/json;charset=UTF-8'
    }
body1={"appKey":"P3ewIzs9gigHVSRLXqvF4xjubPwDz5MZ","pubAppId":175680,
      "fomulars":"[]",
      "groups":"[{\"id\":\"ba8d355e-312a-4985-b4e5-99610721691f\",\"display\":\"用户唯一id\",\"displayEn\":\"SSID\",\"value\":\"$ssid\",\"propType\":1,\"type\":1,\"category\":0,\"dataType\":2},{\"id\":\"adbc222c-8060-4da1-a5f0-c0d9cd68abfe\",\"display\":\"事件发生时间\",\"displayEn\":\"事件发生时间\",\"value\":\"#vp@event_date\",\"propType\":0,\"type\":2,\"virtualSql\":\"event.$sts\",\"category\":0,\"dataType\":3},{\"id\":\"7fa9fa3e-6126-4f68-9629-52c0ec2ce6de\",\"display\":\"国家/地区\",\"displayEn\":\"Country/Region\",\"value\":\"$country\",\"propType\":1,\"type\":1,\"category\":0,\"dataType\":2},{\"id\":\"17f50bca-0b51-4596-bc3e-ef1b6aa5a1db\",\"display\":\"首次广告活动ID\",\"displayEn\":\"First Ad Campaign\",\"value\":\"#utm_campaign\",\"propType\":1,\"type\":1,\"category\":1,\"dataType\":2}]",
      "tz":0,"dateType":2,"lang":"cn",
      "dateValue":"[60,\"Yesterday\"]",
      "events":"[{\"countType\":{\"value\":\"total\",\"name\":\"总次数\",\"nameEn\":\"Total\"},\"display\":\"App 安装后首次启动\",\"displayEn\":\"app intall\",\"eid\":\"#app_install\",\"f\":[],\"id\":\"45524401-0d84-4ce6-b8b1-a15bafc5a777\",\"type\":0}]",
       "filters":"[]"}

body2 ={"appKey":"P3ewIzs9gigHVSRLXqvF4xjubPwDz5MZ","pubAppId":175680,
        "fomulars":"[]",
        "groups":"[{\"id\":\"ba8d355e-312a-4985-b4e5-99610721691f\",\"display\":\"用户唯一id\",\"displayEn\":\"SSID\",\"value\":\"$ssid\",\"propType\":1,\"type\":1,\"category\":0,\"dataType\":2},{\"id\":\"adbc222c-8060-4da1-a5f0-c0d9cd68abfe\",\"display\":\"事件发生时间\",\"displayEn\":\"事件发生时间\",\"value\":\"#vp@event_date\",\"propType\":0,\"type\":2,\"virtualSql\":\"event.$sts\",\"category\":0,\"dataType\":3}]",
        "tz":0,"dateType":2,"lang":"cn",
        "dateValue":"[60,\"Yesterday\"]",
        "events":"[{\"countType\":{\"value\":\"totalProp\",\"name\":\"按...求和(转换为统一币种后收入)\",\"nameEn\":\"Total of Property value(App Currency Revenue)\",\"props\":\"#em_revenue_ex\",\"propType\":1,\"type\":0},\"display\":\"内购成功\",\"displayEn\":\"内购成功\",\"eid\":\"In_appPurchases_BuySuccess\",\"f\":[],\"id\":\"45524401-0d84-4ce6-b8b1-a15bafc5a777\",\"type\":1}]",
        "filters":"[]"}

In [3]:
df_install = requests.post(url, headers=headers, json=body1)

In [4]:
df_install

<Response [200]>

In [5]:
res_install=pd.DataFrame()
for i in  df_install.json()['data']['data']:
    col_list=['ssid','country','#vp@event_date','#utm_campaign']
    data_values = {k: v for k, v in i.items() if k in col_list}
    df=pd.DataFrame([data_values])
    if res_install.shape[0]==0:
        res_install = df
    else:
        res_install = pd.concat((res_install,df))

In [6]:
res_install

,country,#utm_campaign,ssid,#vp@event_date
0,TW,GG-media buy-Android-tw-1026-1.0-lqs-勇者 (21842...,7CE39mwGW5dpecsXGtozjI,1730785232390
0,KR,,4ytksHxGHG06nRYNjw4p9V,1730218974732
0,JP,,6YQhAp8pm8yGS04daX8lLk,1727717912941
0,CN,,2ih21I9RKOSFUQLd5RgW7j,1727857520293
0,ES,,70LZAJSzrdiUB2h2EmBjZ5,1728733959726
...,...,...,...,...
0,JP,GG-media buy-Android-JP-0928-2.5-lxh-勇者 (21759...,nUqxPWbRqqsuWdS6HS7No,1730518817290
0,JP,,3r6nPaEGTQO16KtM7HWZwr,1727519311467
0,JP,GG-media buy-Android-JP-1008-3.0-lxh-勇者 (21792...,5WtTyOhO10XyE7dIgbXHs4,1729343409581
0,JP,,569Vnlv0DZA25MvQBxINS6,1730050125132


In [7]:
df_ng =requests.post(url, headers=headers, json=body2)

In [8]:
res_ng=pd.DataFrame()
for i in  df_ng.json()['data']['data']:
    col_list=['ssid','#vp@event_date','A:内购成功-按...求和(转换为统一币种后收入)']
    data_values = {k: v for k, v in i.items() if k in col_list}
    df=pd.DataFrame([data_values])
    if res_ng.shape[0]==0:
        res_ng = df
    else:
        res_ng = pd.concat((res_ng,df))

In [9]:
res_install['安装时间']=pd.to_datetime(res_install['#vp@event_date'],unit='ms')
res_ng['内购时间']=pd.to_datetime(res_ng['#vp@event_date'],unit='ms')
res_ng['total'] = res_ng['A:内购成功-按...求和(转换为统一币种后收入)']

In [10]:
res=res_install.merge(res_ng[['ssid','内购时间','total']],how='left',on='ssid')

In [11]:
res.head()

,country,#utm_campaign,ssid,#vp@event_date,安装时间,内购时间,total
0,TW,GG-media buy-Android-tw-1026-1.0-lqs-勇者 (21842...,7CE39mwGW5dpecsXGtozjI,1730785232390,2024-11-05 05:40:32.390,NaT,NaN
1,KR,,4ytksHxGHG06nRYNjw4p9V,1730218974732,2024-10-29 16:22:54.732,NaT,NaN
2,JP,,6YQhAp8pm8yGS04daX8lLk,1727717912941,2024-09-30 17:38:32.941,NaT,NaN
3,CN,,2ih21I9RKOSFUQLd5RgW7j,1727857520293,2024-10-02 08:25:20.293,NaT,NaN
4,ES,,70LZAJSzrdiUB2h2EmBjZ5,1728733959726,2024-10-12 11:52:39.726,NaT,NaN


In [12]:
#  计算间隔天数
res['ng_day']=(res['内购时间']-res['安装时间']).dt.days

In [13]:
# 格式化安装时间
res['安装时间'] = res['安装时间'].dt.strftime('%Y-%m-%d')

In [14]:
# 计算不同渠道总安装数
res_cam_install = res.groupby(by=['安装时间','#utm_campaign'],as_index=False)['ssid'].nunique()

In [15]:
# 计算60日总收入
# 总ltv
ltv_df=res.groupby(by=['安装时间','#utm_campaign'],as_index=False)['ssid'].nunique()
ltv_df['安装时间']= pd.to_datetime(ltv_df['安装时间'])
today =(datetime.date.today())
ltv_df['nday'] = (pd.Timestamp(today)  - ltv_df['安装时间']).dt.days

# 总收入
shouru_df=res.groupby(by=['安装时间','#utm_campaign'],as_index=False)['ssid'].nunique()
shouru_df['安装时间']= pd.to_datetime(shouru_df['安装时间'])
shouru_df['nday'] = (pd.Timestamp(today)  - shouru_df['安装时间']).dt.days
for day in range(61):
    df=res.query(f'ng_day<={day}').groupby(by=['安装时间','#utm_campaign'],as_index=False)['total'].sum()
    # 填充nan
    df.fillna(value=0,inplace=True)
    df.columns=['安装时间','#utm_campaign',f'D{day}收入']
    df['安装时间'] =pd.to_datetime(df['安装时间'])
    # ltv
    ltv_df=ltv_df.merge(df,how='left',on=['安装时间','#utm_campaign'])
    # 计算ltv
    ltv_df[f'D{day}-ltv']=ltv_df[f'D{day}收入']/ltv_df['ssid']
    # 保留两位小数
    ltv_df=ltv_df.round(2)
    ltv_df.fillna(value=0,inplace=True)
    # 删除无用列
    ltv_df.drop(columns=[f'D{day}收入'],inplace=True)
    # 判断安装时间是否度过 day 日窗口期 
    ltv_df[f'D{day}-ltv']=ltv_df.apply(lambda x:'-' if x['nday']< (day+1) else x[f'D{day}-ltv'],axis=1)
    #收入---------------------------------------
    shouru_df=shouru_df.merge(df,how='left',on=['安装时间','#utm_campaign'])
    # 保留两位小数
    shouru_df=shouru_df.round(2)
    shouru_df.fillna(value=0,inplace=True)
    # 判断安装时间是否度过 day 日窗口期 
    shouru_df[f'D{day}收入']=shouru_df.apply(lambda x:'-' if x['nday']< (day+1) else x[f'D{day}收入'],axis=1)

In [16]:
ltv_df.sort_values(by=['安装时间','#utm_campaign'],ascending=[True,True],inplace=True)
shouru_df.sort_values(by=['安装时间','#utm_campaign'],ascending=[True,True],inplace=True)

In [17]:
ltv_df.drop(columns=['nday'],inplace=True)
shouru_df.drop(columns=['nday'],inplace=True)

In [18]:
# 写入日期
starttime =(datetime.date.today() - datetime.timedelta(days=60)).strftime('%Y-%m-%d')
print(starttime)
# 写入飞书表格
tab_path = 'R1F0sluEBhHbhqtXJ5BcCiJZnnf'
sheet_path = 'Gxwiv4'
range_list = 'A:A'
df = feishu.read_tab(tab_path, sheet_path, range_list)
if df.shape[0] <= 1:
    new_range = 2
else:
    new_range = list(df.query(f'日期=="{starttime}"').index)[0] + 1
# 写入飞书表格
ltv_df['安装时间']=ltv_df['安装时间'].dt.strftime('%Y-%m-%d')
values = np.array(ltv_df).tolist()
data_len = new_range  + len(values)  -1
feishu.insert_tab(tab_path, sheet_path, f'A{new_range}:BL{data_len}', values)
print('-- 系列回收更新完成')

2024-09-12
t-g104bbagBP5PVGA3ZYQIRBIYXOPR7YTFUNTVK325
t-g104bbagBP5PVGA3ZYQIRBIYXOPR7YTFUNTVK325
success
-- 系列回收更新完成


In [19]:
# 写入日期
starttime =(datetime.date.today() - datetime.timedelta(days=60)).strftime('%Y-%m-%d')
print(starttime)
# 写入飞书表格
tab_path = 'R1F0sluEBhHbhqtXJ5BcCiJZnnf'
sheet_path = 'LUlkG3'
range_list = 'A:A'
df = feishu.read_tab(tab_path, sheet_path, range_list)
if df.shape[0] <= 1:
    new_range = 2
else:
    new_range = list(df.query(f'日期=="{starttime}"').index)[0] + 1
# 写入飞书表格
shouru_df['安装时间']=shouru_df['安装时间'].dt.strftime('%Y-%m-%d')
values = np.array(shouru_df).tolist()
data_len = new_range  + len(values)  -1
feishu.insert_tab(tab_path, sheet_path, f'A{new_range}:BL{data_len}', values)
print('-- 系列回收更新完成')

2024-09-12
t-g104bbagBP5PVGA3ZYQIRBIYXOPR7YTFUNTVK325
t-g104bbagBP5PVGA3ZYQIRBIYXOPR7YTFUNTVK325
success
-- 系列回收更新完成


In [211]:
shouru_df

,安装时间,#utm_campaign,ssid,D0收入,D1收入,D2收入,D3收入,D4收入,D5收入,D6收入,...,D51收入,D52收入,D53收入,D54收入,D55收入,D56收入,D57收入,D58收入,D59收入,D60收入
0,2024-09-05,,192,42.0,43.0,44.0,44.0,44.0,44.0,45.0,...,46.0,46.0,46.0,46.0,46.0,46.0,46.0,46.0,46.0,-
1,2024-09-05,GG-media buy-Android-JP-0831-1.0-lxh (21656829...,143,42.0,58.0,58.0,77.0,92.0,95.0,103.0,...,142.0,142.0,142.0,147.0,150.0,170.0,171.0,171.0,171.0,-
2,2024-09-05,kingrush-1.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-
3,2024-09-06,,98,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-,-
4,2024-09-06,GG-media buy-Android-JP-0831-1.0-lxh (21656829...,254,5.0,11.0,11.0,48.0,49.0,56.0,56.0,...,129.0,129.0,175.0,175.0,176.0,184.0,185.0,188.0,-,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,2024-11-03,Meetsocial_HX-Facebook-JP-AND-240808-MAIA-AAA ...,1,0.0,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
299,2024-11-03,ToB-HK/MO/TW-lxh-0727-2.5,2,0.0,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
300,2024-11-03,ToB-lxh-0726-1.0,2,0.0,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
301,2024-11-03,kingrush-1.0,1,0.0,-,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,-
